# 18 · B3-full: OCR-aware QLoRA trên toàn bộ train ViTextVQA (W10)

**Vấn đề:** Kết quả W09 trên full test 10.028 câu:

| Run | Dữ liệu train | OCR | EM |
|---|---|---|---|
| B2-sub | 8.000 câu (23%) | không | 46,56 |
| B3 | 8.000 câu (23%) | có | 49,10 |
| B2 (W07) | 35.159 câu (100%) | không | 50,26 |

B3 thắng B2-sub có ý nghĩa (p = 1,1 × 10⁻¹⁴) nhưng vẫn **thấp hơn B2** 1,16 điểm. Hội đồng sẽ hỏi: *"Khi đủ dữ liệu, train cùng OCR còn giúp không?"* Câu trả lời quyết định đóng góp của đề tài là **"tốt hơn"** hay chỉ **"hiệu quả dữ liệu"**.

**Thiết kế (luật ghi trong `docs/W10/README.md` TRƯỚC khi chạy):** **B3-full** = công thức B3 **y hệt** (attention r 16 · α 32, OCR VietOCR · v1 · 600 · 0, lr 2e-4, 1 epoch, seed 42, bf16, 4-bit, L4), chỉ đổi tập con 8k → **toàn bộ `train.json`** (cùng file, cùng sha256 với B2 W07). Eval full test có OCR.

| Kết luận (B2 → B3-full, ghép cặp 10.028 câu) | Điều kiện |
|---|---|
| **OCR-AWARE VƯỢT B2 KHI ĐỦ DỮ LIỆU** | EM tăng **và** McNemar p < 0,05 |
| **OCR-AWARE KÉM B2 KHI ĐỦ DỮ LIỆU** | EM giảm **và** p < 0,05 |
| **NGANG B2 KHI ĐỦ DỮ LIỆU** | còn lại → đóng góp chỉ là hiệu quả dữ liệu, ghi đúng như vậy |

Kèm: B3-full so với ViTextBLIP-2 (luật CI95 như W09) và bảng 2×2 *dữ liệu × OCR* (chỉ báo cáo).

| Phần | Việc | Thời gian (L4) |
|---|---|---|
| Train | 2.198 step × ≈ 21,5 s | ≈ 13 giờ |
| Eval | full test 10.028 câu, có OCR | ≈ 2,8 giờ |
| Kết luận | so ghép cặp + bootstrap | ≈ 3–5 phút (CPU) |

**Cần trước:** notebook 10 **Phần D** (T4) đã OCR đủ 11.733 ảnh train. **Chia phiên:** train lưu checkpoint mỗi 100 step (≈ 36 phút); eval ghi mỗi lô 500 câu. Bị ngắt → mở lại notebook, **Run all**: phần đã xong được bỏ qua, phần dở chạy tiếp.

## 1. Thiết lập môi trường

Giống notebook 17 (cùng pin `transformers==5.17.0`, cùng GPU L4). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 18), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/18_qlora_ocr_b3full_vs_b2_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 18'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B2/B3). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn 16 giờ GPU

- `train.json` trên Drive **trùng sha256** với file B2 W07 đã train (35.159 câu) — cùng dữ liệu với mốc so sánh.
- Cấu hình OCR đọc từ `ocr_selection.json` (W08), **không gõ tay**.
- OCR cache **trùng sha256** biên bản Phần D (`W10_ocr_cache_fulltrain`) và phủ đủ **11.733 ảnh train + 3.353 ảnh test**. Chưa có biên bản → dừng, chạy notebook 10 Phần D (T4) trước.
- Ba run full test dùng để so sánh (B2 W07, B2-sub và B3 W09) còn trên Drive, đủ 10.028 câu.
- Selftest 5 script.

*Kỳ vọng: `train.json sha256 = B2 W07 ✅ (35159 câu)`, `paddledet_vietocr | v1 | 600 | 0.0`, `OCR cache = biên bản Phần D ✅`, `train 11733/11733 · test 3353/3353`, 3 dòng `… 10028 câu ✅`, `✅ selftest OK (5 script)`.*

In [ ]:
import json, hashlib, subprocess
import pandas as pd
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
TRAIN = f'{DRIVE_ROOT}/data/{DS}/train.json'
FULL_TEST = f'{DRIVE_ROOT}/data/{DS}/test.json'
OUT = f'{EXP_ROOT}/W10_B3full_vs_B2_testfull_L4_seed42'
SAVE_STEPS = 100        # checkpoint ~36 phút/lần (không ảnh hưởng chữ ký run)
CHUNK = 500

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

# (a) cùng file train với B2 W07
B2_TRAIN_LOG = 'experiments/W07_B2_qlora_r16_qwen25vl_vitextvqa_ep1_bf16_L4_seed42/train_log.json'
b2_meta = json.load(open(B2_TRAIN_LOG, encoding='utf-8'))['meta']
assert sha256_file(TRAIN) == b2_meta['data_file_sha256'], 'train.json trên Drive KHÁC file B2 W07 đã train -> dừng, báo Claude'
print(f"train.json sha256 = B2 W07 ✅ ({b2_meta['num_samples']} câu)")

# (b) cấu hình OCR đóng băng (W08)
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

# (c) OCR cache = biên bản Phần D, phủ đủ train + full test
C_PATH = default_cache_path(DRIVE_ROOT, DS, BACKEND)
MAN_D = f'{EXP_ROOT}/W10_ocr_cache_fulltrain/ocr_cache_manifest.json'
if not os.path.exists(MAN_D):
    raise SystemExit('⏸ Chưa có biên bản OCR Phần D -> chạy notebook 10 Phần D (T4) trước, rồi Run all lại notebook này.')
man_d = json.load(open(MAN_D, encoding='utf-8'))
assert sha256_file(C_PATH) == man_d['sha256_after_w10'], 'OCR cache trên Drive KHÁC biên bản Phần D -> dừng, báo Claude'
print('OCR cache = biên bản Phần D ✅')
IMG = prepare_images(DS, DRIVE_ROOT)
have = set(load_ocr_cache(C_PATH)['images'])
tr, te = set(images_of([TRAIN], IMG)), set(images_of([FULL_TEST], IMG))
print(f'train {len(tr & have)}/{len(tr)} · test {len(te & have)}/{len(te)}')
assert tr <= have and te <= have, 'OCR cache thiếu ảnh -> chạy notebook 10 Phần D'

# (d) các run full test để so sánh
REF = {'B2':    'W07_B2_eval_qlora_r16_ep1_vitextvqa_testfull_bf16_L4_seed42',
       'B2sub': 'W09_B2sub_eval_qlora_r16_sub8k_ep1_vitextvqa_testfull_bf16_L4_seed42',
       'B3':    'W09_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_testfull_bf16_L4_seed42'}
for k, r in REF.items():
    n = len(json.load(open(f'{EXP_ROOT}/{r}/predictions.json', encoding='utf-8')))
    assert n == 10028, (k, n)
    print(f'{k:6s} {r[:40]}… {n} câu ✅')

B3F_TRAIN = 'W10_B3full_qlora_ocr_vietocr_v1_r16_qwen25vl_vitextvqa_full_ep1_bf16_L4_seed42'
B3F_EVAL = 'W10_B3full_eval_qlora_ocr_vietocr_v1_r16_full_ep1_vitextvqa_testfull_bf16_L4_seed42'

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (không chạy eval trên adapter hỏng)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

checks = {'src.adaptation.train_qlora': 'lora override OK', 'src.adaptation.lora_targets': 'lora audit OK',
          'src.eval.b3full': 'b3full rules OK', 'src.eval.run_baseline': 'SELFTEST OK',
          'src.eval.paper_consistency': 'SELFTEST OK'}
for mod, must in checks.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout and 'SELFTEST OK' in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (5 script)')

## 3. Train B3-full (≈ 13 giờ)

Lệnh **y hệt** B3 (notebook 13), chỉ bỏ `--data_file` (mặc định = toàn bộ `train.json`). Sau ≈ 4 phút nạp model, log phải có `LoRA audit: 144 lớp {'llm': 144} | … | tham số LoRA 7,372,800 (tính tay 7372800)` và `lịch: 2198 step tổng`.

Sau khi train, kiểm `train_log.json`: seed 42, đủ 35.159 câu (không loại mẫu nào), cùng sha256 dữ liệu với B2 W07, r 16 / α 32 / attention, LoRA chỉ ở LLM, cấu hình OCR khớp `ocr_selection.json`.

**Giữ máy không ngủ và tab mở.** Bị ngắt thì Run all lại → chạy tiếp từ checkpoint gần nhất.

*Kỳ vọng: `== XONG: 2198 step …`, `✔ B3-full train_log …`. **Điểm dừng 1** (an toàn để tắt máy nếu cần).*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{B3F_TRAIN}/train_log.json'):
    print('⏭ B3-full train: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.adaptation.train_qlora --dataset {DS} --run_name {B3F_TRAIN} '
            f'--epochs 1 --save_steps {SAVE_STEPS} {OCR_ARGS}')

meta = json.load(open(f'{EXP_ROOT}/{B3F_TRAIN}/train_log.json', encoding='utf-8'))['meta']
assert meta['seed'] == 42, meta['seed']
assert meta['data_file_sha256'] == b2_meta['data_file_sha256'] and meta['num_samples'] == b2_meta['num_samples'] == 35159, \
    (meta['num_samples'], meta['num_dropped'])
assert (meta['lora']['r'], meta['lora']['lora_alpha'], meta['lora']['target_modules']) == \
       (16, 32, ['q_proj', 'k_proj', 'v_proj', 'o_proj']), meta['lora']
assert meta['lora_audit']['by_tower'] == {'llm': 144} and meta['trainable_params'] == 7372800, meta['lora_audit']
assert {k: meta['ocr'][k] for k in ('backend', 'template', 'max_chars', 'min_conf')} == \
       {'backend': BACKEND, 'template': TPL, 'max_chars': MAXC, 'min_conf': MINC}, meta['ocr']
for k in ('learning_rate', 'epochs', 'per_device_batch_size', 'gradient_accumulation_steps', 'warmup_ratio'):
    assert meta['training'][k] == b2_meta['training'][k], (k, meta['training'][k], b2_meta['training'][k])
AD_B3F = f'{ADP_ROOT}/{B3F_TRAIN}/epoch_1'
assert os.path.isdir(AD_B3F), f'Thiếu adapter {AD_B3F}'
c = meta['cost']
print(f"✔ B3-full train_log: {meta['num_samples']} câu | {meta['training']['total_steps']} step | "
      f"{c['train_runtime_sec'] / 3600:.2f} h (phiên cuối) | {c['sec_per_step']} s/step | VRAM {c['peak_vram_gb']} GB | "
      f"loss {c['final_loss']} | chạy tiếp từ {meta['resumed_from']}")

## 4. Eval B3-full trên full test 10.028 câu (≈ 2,8 giờ)

Adapter `epoch_1`, prompt **có** dòng OCR (cấu hình đóng băng), giống hệt cách chấm B3 full test ở W09. Mỗi lô 500 câu ghi lên Drive ngay khi xong.

*Kỳ vọng: dòng `OCR: …`, 21 lô, `== KẾT QUẢ: EM=…`. **Điểm dừng 2.***

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{B3F_EVAL}/metrics.json'):
    print('⏭ B3-full eval: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --data_file {FULL_TEST} --run_name {B3F_EVAL} '
            f'--chunk {CHUNK} --adapter {AD_B3F} {OCR_ARGS}')
m = json.load(open(f'{EXP_ROOT}/{B3F_EVAL}/metrics.json', encoding='utf-8'))
print(f"✔ B3-full full test: EM={m['metrics']['exact_match']:.4f} ANLS={m['metrics']['anls']:.4f} "
      f"F1={m['metrics']['token_f1']:.4f}  (B2 W07: EM 0.5026 ANLS 0.7202)")

## 5. Kết luận theo luật ghi trước

`src/eval/b3full.py` ghép cặp 4 run trên cùng 10.028 câu:
1. **So sánh chính B2 → B3-full** → một trong 3 kết luận ở đầu notebook.
2. **B3-full so với ViTextBLIP-2** (EM 25,48 · F1 53,95): "CAO HƠN" chỉ khi cận dưới CI95 bootstrap > số công bố.
3. **Chỉ báo cáo:** bảng 2×2 *dữ liệu (23% / 100%) × OCR (không / có)*, 4 hiệu ứng đơn và tương tác theo câu.

Bootstrap 10.000 lần trên 10.028 câu chạy bằng CPU, mất ≈ 3–5 phút — **đừng dừng cell**.

*Kỳ vọng: bảng 2×2, 4 dòng hiệu ứng, 2 dòng tương tác, dòng `➜ KẾT LUẬN THEO LUẬT (B2 -> B3full): …`, bảng so với ViTextBLIP-2, file `b3full_summary.json`.*

In [ ]:
from src.eval import paper_consistency as pc
os.makedirs(OUT, exist_ok=True)
run_cmd(f"python -m src.eval.b3full --b2sub {EXP_ROOT}/{REF['B2sub']} --b3 {EXP_ROOT}/{REF['B3']} "
        f"--b2 {EXP_ROOT}/{REF['B2']} --b3full {EXP_ROOT}/{B3F_EVAL} --out {OUT}/b3full_summary.json")

s = json.load(open(f'{OUT}/b3full_summary.json', encoding='utf-8'))
P = json.load(open(f'{EXP_ROOT}/{B3F_EVAL}/predictions.json', encoding='utf-8'))
sota = pc.vitextblip2_report(P)
display(pd.DataFrame([{'metric': met, 'B3-full': sota[met]['ours_pct'], 'CI95': sota[met]['ours_ci95_pct'],
                       'ViTextBLIP-2 (paper)': sota[met]['published_pct'], 'kết luận': sota[met]['verdict']}
                      for met in ('EM', 'F1')]))

rows = []
for lab, run in (('B2-sub (23%, không OCR)', REF['B2sub']), ('B3 (23%, OCR)', REF['B3']),
                 ('B2 W07 (100%, không OCR)', REF['B2']), ('B3-full (100%, OCR)', B3F_EVAL)):
    mm = json.load(open(f'{EXP_ROOT}/{run}/metrics.json', encoding='utf-8'))
    rows.append({'run': lab, 'EM': round(100 * mm['metrics']['exact_match'], 2), 'ANLS': round(100 * mm['metrics']['anls'], 2),
                 'F1 (phụ)': round(100 * mm['metrics']['token_f1'], 2), 's/câu': mm['meta'].get('sec_per_sample')})
display(pd.DataFrame(rows))
cost = [{'run': 'B2 W07', 'giờ train': round(b2_meta['cost']['train_runtime_sec'] / 3600, 2), 's/step': b2_meta['cost']['sec_per_step'],
         'VRAM đỉnh (GB)': b2_meta['cost']['peak_vram_gb'], 'loss cuối': b2_meta['cost']['final_loss'], 'chạy tiếp từ': b2_meta['resumed_from']},
        {'run': 'B3-full', 'giờ train': round(meta['cost']['train_runtime_sec'] / 3600, 2), 's/step': meta['cost']['sec_per_step'],
         'VRAM đỉnh (GB)': meta['cost']['peak_vram_gb'], 'loss cuối': meta['cost']['final_loss'], 'chạy tiếp từ': meta['resumed_from']}]
display(pd.DataFrame(cost))
s.update({'vs_vitextblip2': sota, 'reference': pc.TABLE6_VITEXTBLIP2, 'source': pc.PAPER_SOURCE, 'table': rows,
          'cost': cost, 'ocr_config': sel, 'ocr_cache_manifest': MAN_D.replace(DRIVE_ROOT + '/', ''), 'gpu': GPU,
          'rule_doc': 'docs/W10/README.md mục Luật B3-full'})
json.dump(s, open(f'{OUT}/b3full_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
print('->', f'{OUT}/b3full_summary.json')

## 6. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Notebook dừng sớm vì lỗi thì máy **không** tự trả → tự bấm **Runtime → Disconnect and delete runtime** sau khi đọc lỗi.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy L4 …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy L4 để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 7. ⏸ DỪNG

Gửi Claude output **mục 5** (và dòng `✔ B3-full train_log …` ở mục 3 nếu còn trong tab). Nếu tab đã đóng mất output: không sao — mọi kết quả nằm trong `experiments/W10_B3full*` trên Drive, Cursor chép và kiểm lại được.